In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.special import gammaln
import sys
from scipy.ndimage import gaussian_filter1d
from scipy.stats import wilcoxon
from concurrent import futures 
# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import importlib
import session_class
import cell_analysis
import multi_session_pca
importlib.reload(session_class)
importlib.reload(cell_analysis)
importlib.reload(multi_session_pca)

from session_class import Session
from cell_analysis import Cell
from multi_session_pca import MultiSessionPCA

import holoviews as hv
from holoviews import opts
from bokeh.io import output_notebook
import hvplot.pandas  # noqa: F401
output_notebook()
hv.extension('bokeh')

print("Imports loaded successfully!")

Loading BokehJS ...

Imports loaded successfully!


In [40]:
base_path = Path.cwd().parents[1] / 'data' / 'unified_cell_trial_data'
pickle_file = base_path / 'msn_fiona_cell_trial_data.pkl'

# Load MSN cell database
cell_df = pd.read_pickle(pickle_file)


In [41]:
# Configuration
SSRT = 50  # Stop Signal Reaction Time in ms
SMOOTH_SIGMA = 15  # Smoothing sigma for gaussian filter (ms)
MIN_GRADE = 8   # Minimum cell quality grade

# Epoch parameters
BIN_SIZE = 1      # ms
EPOCH_WINDOW = [-200, 600] # ms around alignment event

"""
From Indra et al., 2020, methods section page 5;380:
The recording cylinder was implanted on the left hemisphere for monkey F(iona)
"""
CONTRA_DIR = 0  # Contra direction for monkey Fiona

print(f"Configuration set: SSRT={SSRT}ms, Grade>={MIN_GRADE}")


Configuration set: SSRT=50ms, Grade>=8


In [42]:
cell_id = 1547
cell_df['cell_ID'] ==cell_id
cell = Cell(cell_df[cell_df['cell_ID'] == cell_id])

print(cell.data['trial_number'].nunique() == cell.data.shape[0])
print(cell.data.shape)

cell.data.columns

True
(685, 27)


Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')

In [43]:
cell.align_spikes_to_event(
    alignment_point='first_relevant_saccade', verbose=True
)

cell.align_spikes_to_event(
    alignment_point='go_cue', verbose=True
)

cell.data.columns

Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session',
       'spikes_aligned_to_first_relevant_saccade', 'spikes_aligned_to_go_cue'],
      dtype='object')

In [44]:
cell.plot_raster(
    epok=[-500, 200],
    alignment_point='first_relevant_saccade',
    # alignment_point='go_cue',
    trial_type='GO',
    success_only=False,
    # direction=contra_dir,
)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .HeatMap.I :HeatMap   [time,index]   (value)

In [45]:
def get_trial_spikes_for_Indra_condition_1(trial: pd.Series):
    """Count the amount of spikes"""
    assert trial['type'] == 'GO'
    assert trial['dir'] == CONTRA_DIR
    assert trial['trial_failed'] == False

    baseline_spikes_count = (
        (trial['spikes_aligned_to_go_cue'] >= -300) & 
        (trial['spikes_aligned_to_go_cue'] <= 0)
    ).sum()

    saccade_spikes_count = (
        (trial['spikes_aligned_to_first_relevant_saccade'] >= -100) & 
        (trial['spikes_aligned_to_first_relevant_saccade'] <= 100)
    ).sum()

    return baseline_spikes_count, saccade_spikes_count

def get_cell_spike_counts_for_Indra_condition_1(cell: Cell):
    """Count the amount of spikes for all legal trials of the cell"""
    cell.align_spikes_to_event(
        alignment_point='first_relevant_saccade', verbose=True
    )

    cell.align_spikes_to_event(
        alignment_point='go_cue', verbose=True
    )

    tmp_df = cell.data[
        (cell.data['type'] == 'GO') &
        (cell.data['dir'] == CONTRA_DIR) &
        (cell.data['trial_failed'] == False)
    ]

    spike_counts = tmp_df.apply(get_trial_spikes_for_Indra_condition_1, axis=1, result_type='expand').to_numpy()
    base_line_counts = spike_counts[:, 0]
    saccade_counts = spike_counts[:, 1]

    return base_line_counts, saccade_counts

def run_signed_rank_test_for_cell_Indra_condition_1(cell: Cell):
    """
    "GO" neurons were defined based on two criteria. The first: 
    (1) a significant firing rate increase around contralateral saccade onset 
        (-100 ms to +100 ms from saccade onset versus baseline from -300 ms to go) 
        on the go trials (one-tailed Wilcoxon signed-rank, P ≤ 0.05)
    """
    base_line_counts, saccade_counts = get_cell_spike_counts_for_Indra_condition_1(cell)

    stat, p_value = wilcoxon(
        base_line_counts,
        saccade_counts,
        alternative='greater'
    )
    return stat, p_value



trial = cell.data.iloc[212]
print(trial['spikes_aligned_to_go_cue'])
print(get_trial_spikes_for_Indra_condition_1(trial))

cell_dist = get_cell_spike_counts_for_Indra_condition_1(cell)
print(f"Cell spike counts shape: {cell_dist[0].shape}, {cell_dist[1].shape}")

stat, p_value = run_signed_rank_test_for_cell_Indra_condition_1(cell)
print(f"Wilcoxon signed-rank test result: stat={stat}, p-value={p_value}")


[-842.57 -114.08  -15.22]
(np.int64(2), np.int64(0))
Cell spike counts shape: (197,), (197,)
Wilcoxon signed-rank test result: stat=3419.0, p-value=1.7246074218501988e-13


In [ ]:
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

def process_cell(cell_id: int) -> tuple:
    """Process a single cell and return its signed rank test results"""
    try:
        cell_data = cell_df[cell_df['cell_ID'] == cell_id]
        cell_obj = Cell(cell_data)
        stat, p_value = run_signed_rank_test_for_cell_Indra_condition_1(cell_obj)
        return cell_id, stat, p_value, None
    except Exception as e:
        return cell_id, np.nan, np.nan, str(e)

# Get unique cell IDs
unique_cell_ids = cell_df['cell_ID'].unique()
print(f"Processing {len(unique_cell_ids)} cells...")

# Process cells in parallel
results = []
with futures.ProcessPoolExecutor() as executor:
    future_to_cell = {executor.submit(process_cell, cell_id): cell_id for cell_id in unique_cell_ids}
    for future in tqdm(futures.as_completed(future_to_cell), total=len(unique_cell_ids), desc="Processing Cells"):
        result = future.result()
        results.append(result)

# Create DataFrame from results
results_df = pd.DataFrame(results, columns=['cell_ID', 'stat', 'p_value', 'error'])
results_df = results_df.sort_values('cell_ID').reset_index(drop=True)

# Apply Holm-Bonferroni correction (only for non-NaN p-values)
valid_mask = ~results_df['p_value'].isna()
results_df['p_value_holm_bonferroni'] = np.nan

if valid_mask.sum() > 0:
    reject, pvals_corrected, _, _ = multipletests(
        results_df.loc[valid_mask, 'p_value'], 
        method='holm'
    )
    results_df.loc[valid_mask, 'p_value_holm_bonferroni'] = pvals_corrected
    results_df.loc[valid_mask, 'significant_holm'] = reject

print(f"\nCompleted! {valid_mask.sum()} cells processed successfully, {(~valid_mask).sum()} errors")
print(f"Significant cells (Holm-Bonferroni corrected, p<0.05): {results_df['significant_holm'].sum()}")

significant_cell_ids = set(results_df[results_df['significant_holm'] == True]['cell_ID'])
# print(f"Number of significant cells: {len(significant_cell_ids)}")

results_df.head(10)

Processing 1414 cells...


Processing Cells: 100%|██████████| 1414/1414 [00:01<00:00, 1300.34it/s]



Completed! 1413 cells processed successfully, 1 errors
Significant cells (Holm-Bonferroni corrected, p<0.05): 515
Number of significant cells: 515


,cell_ID,stat,p_value,error,p_value_holm_bonferroni,significant_holm
0,11,180.0,8.808510e-04,None,6.720893e-01,False
1,15,136.0,3.547483e-02,None,1.000000e+00,False
2,16,532.0,3.965962e-03,None,1.000000e+00,False
3,17,2648.0,4.113529e-10,None,5.026733e-07,True
4,18,522.0,1.632052e-01,None,1.000000e+00,False
5,19,285.0,1.115087e-01,None,1.000000e+00,False
6,20,30.0,6.184877e-01,None,1.000000e+00,False
7,21,1296.0,1.290087e-03,None,9.572446e-01,False
8,23,416.0,5.503126e-03,None,1.000000e+00,False
9,24,128.5,2.042682e-02,None,1.000000e+00,False


In [ ]:
"""
(2) a significant decrease in correct stop trials vs. slow go in the contralateral direction (in
0-400 ms bin from stop signal onset) (one-tailed Wilcoxon signed-rank, P ≤ 0.05)
"""

def get_trial_spikes_for_Indra_condition_2(trial: pd.Series):
    """Count the amount of spikes"""
    assert trial['type'] == 'GO'
    assert trial['dir'] == CONTRA_DIR
    assert trial['trial_failed'] == False

    baseline_spikes_count = (
        (trial['spikes_aligned_to_go_cue'] >= -300) & 
        (trial['spikes_aligned_to_go_cue'] <= 0)
    ).sum()

    saccade_spikes_count = (
        (trial['spikes_aligned_to_first_relevant_saccade'] >= -100) & 
        (trial['spikes_aligned_to_first_relevant_saccade'] <= 100)
    ).sum()

    return baseline_spikes_count, saccade_spikes_count